<style>
@font-face {
  font-family: 'Roboto';
  font-style: normal;
  font-weight: 600;
  src: local('Roboto Semi-Bold'), local('Roboto-SemiBold'), url('assets/Roboto-SemiBold.ttf') format('truetype');
}
</style>

<div align="center">
  <img src="assets/Jupyter_AIKit_logo.svg" width="600">
</div>

<h2 style="color:#FFFFFF; text-align:center; font-family:'Roboto', sans-serif; font-size:24px; font-weight:600; letter-spacing:0.08em;">LORA FINE-TUNING WITH LLAMAFACTORY</h2>

<p>LoRA is a lightweight fine-tuning method for large language models that injects small, trainable adapter layers into the model instead of updating all parameters. This makes training far more memory and compute-efficient, while reducing the risk of significant forgetting compared to full fine-tuning. However, LoRA adapters may sometimes trade off a bit of peak accuracy for efficiency and flexibility.</p>

<p>Check out this article for more information: 
<a href="https://medium.com/@kailash.thiyagarajan/fine-tuning-large-language-models-with-lora-demystifying-efficient-adaptation-25fa0a389075">
Fine-Tuning Large Language Models with LORA: Demystifying Efficient Adaptation</a></p>

<!-- Modules Overview -->
<div style="border-left:4px solid #44D62C; padding:12px 16px; background-color:transparent; margin:14px 0; font-size:14px; border-radius:6px;">
  <b style="color:#44D62C;">Note:</b>  
  This guide uses the following key modules and tools:
  <ul style="margin:8px 0 0 18px;">
    <li><code>llamafactory-cli</code> – streamlined CLI for fine-tuning with optimized defaults and configuration management.</li>
    <li><code>rzr-aikit</code> – Razer CLI to download models, serve fine-tuned models with LoRA adapters, run generation tests and control model deployment.</li>
  </ul>
  <b style="color:#44D62C;">Dependencies:</b> LlamaFactory internally uses <code>transformers</code> and <code>datasets</code> for model handling and data processing.
</div>

This guide uses the model <a href="https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct" style="color:#44D62C;">Qwen/Qwen2.5-0.5B-Instruct</a> with the <a href="https://huggingface.co/datasets/vicgalle/alpaca-gpt4" style="color:#44D62C;">Alpaca-GPT4</a> dataset for instruction-following fine-tuning.

<h3 style="color:#44D62C; text-align:left;">📥 1. Download the Base Model</h3>

Before starting fine-tuning, ensure the base model is available locally. `rzr-aikit` provides convenient model management capabilities for downloading, caching, and organizing models from Hugging Face Hub.

In [ ]:
rzr-aikit model download Qwen/Qwen2.5-0.5B-Instruct

<h3 style="color:#44D62C; text-align:left;">📊 2. Dataset Configuration</h3>

LlamaFactory uses JSON configuration files to define dataset parameters, including data sources, formatting templates, and column mappings. This approach provides flexibility in handling various dataset formats.

In [ ]:
# Create data directory and dataset configuration
mkdir -p ~/fine-tuning/data
cat > ~/fine-tuning/data/dataset_info.json << 'JSON'
{
  "alpaca_gpt4": {
    "hf_hub_url": "vicgalle/alpaca-gpt4",
    "formatting": "alpaca",
    "columns": {
      "prompt": "instruction",
      "query": "input",
      "response": "output"
    }
  }
}
JSON

<h3 style="color:#44D62C; text-align:left;">⚙️ 3. LoRA Configuration Overview</h3>

The LoRA (Low-Rank Adaptation) configuration defines how adapter layers are integrated into the base model:

<ul style="margin:8px 0 0 18px;">
  <li><b style="color:#44D62C;">Rank (r=8):</b> Dimension of low-rank matrices, controlling adapter capacity</li>
  <li><b style="color:#44D62C;">Alpha (α=16):</b> Scaling factor for adapter outputs, typically 1-2x the rank</li>
  <li><b style="color:#44D62C;">Dropout (0.05):</b> Regularization to prevent overfitting in adapter layers</li>
  <li><b style="color:#44D62C;">Target Modules:</b> Attention projection layers (q_proj, k_proj, v_proj, o_proj)</li>
</ul>

<div style="border-left:4px solid #44D62C; padding:12px 16px; background-color:transparent; margin:14px 0; font-size:14px; border-radius:6px;">
  <b style="color:#44D62C;">Performance Tip:</b>  
  Lower rank values (4-16) reduce memory usage but may limit adaptation capacity. Higher alpha values increase the influence of LoRA layers on the base model.
</div>

<h3 style="color:#44D62C; text-align:left;">🔧 4. Fine-Tuning with LlamaFactory</h3>

LlamaFactory provides a CLI interface for LoRA fine-tuning with optimized defaults for various model architectures and training scenarios.

In [ ]:
# Basic Training Configuration
llamafactory-cli train \
  --stage sft \
  --do_train \
  --model_name_or_path Qwen/Qwen2.5-0.5B-Instruct \
  --template qwen \
  --finetuning_type lora \
  --lora_rank 8 \
  --lora_alpha 16 \
  --lora_dropout 0.05 \
  --lora_target q_proj,k_proj,v_proj,o_proj \
  --dataset alpaca_gpt4 \
  --dataset_dir ~/fine-tuning/data \
  --output_dir ~/fine-tuning/adapters/lora_alpaca_gpt4 \
  --overwrite_output_dir \
  --num_train_epochs 1 \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 8 \
  --learning_rate 2e-4 \
  --warmup_ratio 0.03 \
  --cutoff_len 768 \
  --packing True \
  --max_samples 2000 \
  --eval_strategy no \
  --logging_strategy steps \
  --logging_steps 50 \
  --save_strategy no \
  --gradient_checkpointing \
  --group_by_length \
  --fp16

<h4 style="color:#44D62C; text-align:left;">📋 Command Parameters Explained</h4>

<div style="border-left:4px solid #44D62C; padding:12px 16px; background-color:transparent; margin:14px 0; font-size:14px; border-radius:6px;">
  <b style="color:#44D62C;">Basic Training Configuration:</b>
  <ul style="margin:8px 0 0 18px;">
    <li><code>--stage sft</code> - Supervised fine-tuning stage</li>
    <li><code>--do_train</code> - Enable training mode</li>
    <li><code>--model_name_or_path Qwen/Qwen2.5-0.5B-Instruct</code> - Base model identifier</li>
    <li><code>--template qwen</code> - Chat template for Qwen models</li>
  </ul>
  
  <b style="color:#44D62C;">LoRA Adapter Configuration:</b>
  <ul style="margin:8px 0 0 18px;">
    <li><code>--finetuning_type lora</code> - Use LoRA fine-tuning method</li>
    <li><code>--lora_rank 8</code> - Low-rank dimension (controls adapter size)</li>
    <li><code>--lora_alpha 16</code> - Scaling factor for LoRA outputs</li>
    <li><code>--lora_dropout 0.05</code> - Dropout rate in LoRA layers</li>
    <li><code>--lora_target q_proj,k_proj,v_proj,o_proj</code> - Target attention projection layers</li>
  </ul>
  
  <b style="color:#44D62C;">Dataset and Output Configuration:</b>
  <ul style="margin:8px 0 0 18px;">
    <li><code>--dataset alpaca_gpt4</code> - Dataset name from dataset_info.json</li>
    <li><code>--dataset_dir ~/fine-tuning/data</code> - Directory containing dataset configs</li>
    <li><code>--output_dir ~/fine-tuning/adapters/lora_alpaca_gpt4</code> - Output directory for adapters</li>
    <li><code>--overwrite_output_dir</code> - Overwrite existing output directory</li>
  </ul>
  
  <b style="color:#44D62C;">Training Hyperparameters:</b>
  <ul style="margin:8px 0 0 18px;">
    <li><code>--num_train_epochs 1</code> - Number of training epochs</li>
    <li><code>--per_device_train_batch_size 1</code> - Batch size per GPU device</li>
    <li><code>--gradient_accumulation_steps 8</code> - Steps to accumulate gradients (effective batch = 1×8)</li>
    <li><code>--learning_rate 2e-4</code> - Initial learning rate</li>
    <li><code>--warmup_ratio 0.03</code> - Warmup ratio for learning rate scheduling</li>
    <li><code>--cutoff_len 768</code> - Maximum sequence length</li>
    <li><code>--packing True</code> - Pack multiple samples into sequences</li>
    <li><code>--max_samples 2000</code> - Limit training samples for faster iteration</li>
  </ul>
  
  <b style="color:#44D62C;">Logging and Evaluation:</b>
  <ul style="margin:8px 0 0 18px;">
    <li><code>--eval_strategy no</code> - Disable evaluation during training</li>
    <li><code>--logging_strategy steps</code> - Log metrics every N steps</li>
    <li><code>--logging_steps 50</code> - Log every 50 training steps</li>
    <li><code>--save_strategy no</code> - Don't save intermediate checkpoints</li>
  </ul>
  
  <b style="color:#44D62C;">Memory and Performance Optimizations:</b>
  <ul style="margin:8px 0 0 18px;">
    <li><code>--gradient_checkpointing</code> - Trade compute for memory (slower but less VRAM)</li>
    <li><code>--group_by_length</code> - Group samples by length for efficiency</li>
    <li><code>--fp16</code> - Use 16-bit floating point precision</li>
  </ul>
</div>

<h3 style="color:#44D62C; text-align:left;">🚀 5. Deploy Model with LoRA Adapters</h3>

Use VLLM to serve the base model with integrated LoRA adapters for high-performance inference. The server supports multiple LoRA adapters simultaneously, allowing dynamic model switching during inference.

In [ ]:
nohup vllm serve Qwen/Qwen2.5-0.5B-Instruct \
  --enable-lora \
  --lora-modules alpaca=~/fine-tuning/adapters/lora_alpaca_gpt4 \
  --gpu-memory-utilization 0.7 \
  --max-model-len 2048 \
  --enforce-eager \
  > ~/vllm.log 2>&1 &

In [ ]:
tail -f ~/vllm.log \
 | sed -n 'p;/Application startup complete\./q'

<h3 style="color:#44D62C; text-align:left;">🧪 6. Test Base Model Performance</h3>

Generate responses using the base model (without LoRA adapters) to establish baseline performance for comparison with the fine-tuned version.

In [ ]:
rzr-aikit model generate --model Qwen/Qwen2.5-0.5B-Instruct "Explain the concept of machine learning in simple terms."

<h3 style="color:#44D62C; text-align:left;">✨ 7. Test Fine-Tuned Model Performance</h3>

Generate responses using the fine-tuned LoRA adapter to evaluate the impact of instruction-following training on model behavior and response quality.

In [ ]:
rzr-aikit model generate --model alpaca "Explain the concept of machine learning in simple terms."

Once testing is complete, stop the model inference process to free up system resources.

In [ ]:
rzr-aikit model stop